In [1]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic
from pathlib import Path

load_dotenv()

client = Anthropic(
    default_headers={
        "anthropic-beta": "code-execution-2025-08-25, files-api-2025-04-14"
    }
)
model = "claude-sonnet-4-5-20250929"

In [8]:
# Helper functions
import json

from anthropic.types import Message


def add_user_message(messages, message, cache=False):
    user_message = {
        "role": "user",
    }

    if isinstance(message, Message):
        user_message["content"] = message.content
    elif isinstance(message, str) and cache:
        user_message["content"] = [
            {
                "type": "text",
                "text": message,
                "cache_control": {"type": "ephemeral"},
            }
        ]
    else:
        user_message["content"] = message

    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def describe_block(block):
    if block.type == "server_tool_use":
        return f"[{block.name}] input: {json.dumps(block.input)}"

    if block.type == "code_execution_tool_result":
        result = block.content
        if result.type == "code_execution_tool_result_error":
            return f"[code_execution_tool_result] error: {result.error_code}"
        return (
            f"[code_execution_tool_result] return_code={result.return_code} "
            f"stdout={result.stdout!r} stderr={result.stderr!r}"
        )

    if block.type == "thinking":
        return f"[thinking] {block.thinking}"

    return f"[{block.type}] {block}"


def chat(
    messages,
    system=None,
    stop_sequences=[],
    tools=None,
    thinking=False,
    thinking_budget=2000,
    max_tokens=4000,
    stream=True,
):
    params = {
        "model": model,
        "max_tokens": max_tokens,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if thinking:
        params["thinking"] = {
            "type": "enabled",
            "budget_tokens": thinking_budget,
        }

    if tools:
        tools_copy = tools.copy()
        tools_copy[-1] = {**tools_copy[-1], "cache_control": {"type": "ephemeral"}}
        params["tools"] = tools_copy

    if system:
        params["system"] = [
            {
                "type": "text",
                "text": system,
                "cache_control": {"type": "ephemeral"},
            }
        ]

    if not stream:
        return client.messages.create(**params)

    with client.messages.stream(**params) as message_stream:
        for event in message_stream:
            if event.type == "text":
                print(event.text, end="", flush=True)
            elif event.type == "content_block_stop" and event.content_block.type != "text":
                print(f"\n{describe_block(event.content_block)}")
        print()
        return message_stream.get_final_message()


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])


def upload(file_path):
    path = Path(file_path)
    extension = path.suffix.lower()

    mime_type_map = {
        ".pdf": "application/pdf",
        ".txt": "text/plain",
        ".md": "text/plain",
        ".py": "text/plain",
        ".js": "text/plain",
        ".html": "text/plain",
        ".css": "text/plain",
        ".csv": "text/csv",
        ".json": "application/json",
        ".xml": "application/xml",
        ".xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
        ".xls": "application/vnd.ms-excel",
        ".jpeg": "image/jpeg",
        ".jpg": "image/jpeg",
        ".png": "image/png",
        ".gif": "image/gif",
        ".webp": "image/webp",
    }

    mime_type = mime_type_map.get(extension)

    if not mime_type:
        raise ValueError(f"Unknown mimetype for extension: {extension}")
    filename = path.name

    with open(file_path, "rb") as file:
        return client.beta.files.upload(file=(filename, file, mime_type))


def list_files():
    return client.beta.files.list()


def delete_file(id):
    return client.beta.files.delete(id)


def download_file(id, filename=None):
    file_content = client.beta.files.download(id)

    if not filename:
        filename = get_metadata(id).filename

    download_dir = Path("./tmp")
    download_dir.mkdir(parents=True, exist_ok=True)
    file_content.write_to_file(download_dir / filename)


def get_metadata(id):
    return client.beta.files.retrieve_metadata(id)

In [3]:
file_metadata = upload("./csv/streaming.csv")
file_metadata

BetaFileMetadata(id='file_012G9gC9DbEVFEDS7TniyRje', created_at=datetime.datetime(2026, 9, 23, 14, 26, 8, 197763, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, expires_at=None, scope=None)

In [4]:
messages = []

add_user_message(
    messages,
    [
        {
            "type": "text",
            "text": """
Run a detailed analysis to determine major drivers of churn.
Your final output should include at least one detailed plot summarizing your findings.

Critical note: Every time you execute code, you're starting with a completely clean slate. 
No variables or library imports from previous executions exist. You need to redeclare/reimport all variables/libraries.
            """,
        },
        {"type": "container_upload", "file_id": file_metadata.id},
    ],
)

chat(messages, tools=[{"type": "code_execution_20250825", "name": "code_execution"}], max_tokens=20000)

I'll analyze the streaming.csv file to determine the major drivers of churn and create detailed visualizations. Let me start by exploring the data.
[bash_code_execution] input: {"command": "cd $INPUT_DIR && head -20 streaming.csv && wc -l streaming.csv"}

[bash_code_execution_tool_result] BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,Comedy,5,15,32.6,3,7.99,0\nUSER_00002,Premium,41.4,Drama,5,9,45.7,3,17.99,0\nUSER_00003,Standard,33.6,Action,1,7,32.3,4,12.99,1\nUSER_00004,Standard,115.6,Action,12,33,57.3,1,12.99,0\nUSER_00005,Basic,93.8,Documentary,9,27,90.0,2,7.99,1\nUSER_00006,Basic,105.6,Romance,10,27,80.0,2,7.99,0\nUSER_00007,Basic,106.6,Thriller,8,23,53.1,3,7.99,0\nUSER_00008,Prem

ParsedMessage(id='msg_011CfLXUCpLtvxxignLFC5vu', container=Container(id='container_01MM6hBcDt8CtFoNPRmkDk1J', expires_at=datetime.datetime(2026, 9, 23, 15, 28, 29, 511761, tzinfo=TzInfo(0)), skills=None), content=[ParsedTextBlock(citations=None, text="I'll analyze the streaming.csv file to determine the major drivers of churn and create detailed visualizations. Let me start by exploring the data.", type='text', parsed_output=None), ServerToolUseBlock(id='srvtoolu_01VmWjFTHTrkTPQ2LRX6AQvm', caller=None, input={'command': 'cd $INPUT_DIR && head -20 streaming.csv && wc -l streaming.csv'}, name='bash_code_execution', type='server_tool_use'), BashCodeExecutionToolResultBlock(content=BashCodeExecutionResultBlock(content=[], return_code=0, stderr='', stdout='UserID,SubscriptionTier,TotalViewingHoursLastMonth,TopGenre,BingeWatchingSessionsLastMonth,NumberOfUniqueTitlesWatchedLastMonth,AverageSessionDurationMinutes,CustomerServiceInteractionsLastYear,MonthlyCost,Churned\nUSER_00001,Basic,47.9,C

In [5]:
list_files()

SyncPageCursor[BetaFileMetadata](data=[BetaFileMetadata(id='file_01RmKoaa5D6tbYk2UxR29VYW', created_at=datetime.datetime(2026, 9, 23, 14, 28, 26, 421114, tzinfo=datetime.timezone.utc), filename='churn_analysis_report.txt', mime_type='text/plain', size_bytes=4606, type='file', downloadable=True, expires_at=None, scope=None), BetaFileMetadata(id='file_01SnL5rb1DjTD1dbVS4Q7Eu4', created_at=datetime.datetime(2026, 9, 23, 14, 27, 44, 212537, tzinfo=datetime.timezone.utc), filename='churn_analysis_detailed.png', mime_type='image/png', size_bytes=966705, type='file', downloadable=True, expires_at=None, scope=None), BetaFileMetadata(id='file_012G9gC9DbEVFEDS7TniyRje', created_at=datetime.datetime(2026, 9, 23, 14, 26, 8, 197763, tzinfo=datetime.timezone.utc), filename='streaming.csv', mime_type='text/csv', size_bytes=25733, type='file', downloadable=False, expires_at=None, scope=None), BetaFileMetadata(id='file_01NnjetML1ZQhoAqY8G9rXyD', created_at=datetime.datetime(2026, 9, 23, 14, 20, 21, 720

In [12]:
download_file("file_01SnL5rb1DjTD1dbVS4Q7Eu4")